# Setup YOLO26

Pip install `ultralytics` และ [dependencies](https://github.com/ultralytics/ultralytics/blob/main/requirements.txt) แล้วตรวจสอบ software และ hardware.

In [ ]:
%pip install -q "ultralytics>=8.4.0"
import ultralytics
ultralytics.checks()

# 1. Predict (Test Setup YOLO26)

YOLO26s สามารถใช้งานผ่าน Command Line Interface (CLI) ด้วยคำสั่ง `yolo` ได้หลายงานและหลายโหมด และรับ argument เพิ่มเติมได้ เช่น `imgsz=640` ดูรายการ `yolo` [arguments](https://docs.ultralytics.com/usage/cfg/) ทั้งหมดและรายละเอียดอื่นๆ ได้ที่ [YOLO26 Predict Docs](https://docs.ultralytics.com/modes/predict/).

In [ ]:
# Run inference on an image with YOLO26s
!yolo predict model=yolo26s.pt source='https://ultralytics.com/images/bus.jpg'

# 2. Add Zip Data Export YOLO26 for Roboflow

นำไฟล์ `.zip` ที่ดาวน์โหลดมาจาก Roboflow มาอัพโหลดที่นี่ (ตามขั้นตอน **Download Dataset** ที่สอนไว้ก่อนหน้า: กดปุ่ม Download Dataset → เลือกฟอร์แมต **YOLO26** → เลือก **Download zip to computer** → กด Continue จะได้ไฟล์ zip ลงเครื่องของคุณ)

In [ ]:
import zipfile
from google.colab import files

%cd /content
print("เลือกไฟล์ .zip ที่ดาวน์โหลดมาจาก Roboflow (ฟอร์แมต YOLO26)")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("/content/")

print(f"แตกไฟล์ {zip_filename} เรียบร้อยแล้ว")

# 3. Train Data Roboflow

In [ ]:
# Train YOLO26s บนชุดข้อมูลที่แตกไฟล์ zip จาก Roboflow ไว้ที่ /content
!yolo task=detect mode=train model=yolo26s.pt data=data.yaml epochs=100 imgsz=640 plots=True

# 4. Test Model Training

In [ ]:
!yolo detect predict model=/content/runs/detect/train/weights/best.pt conf=0.8 imgsz=640 source='/content/test/images/'

# 5. ทดสอบโมเดลกับวิดีโอจาก Google Drive

ดาวน์โหลดวิดีโอทดสอบจากลิงก์ Google Drive ที่กำหนด (ไฟล์ต้องตั้งค่าแชร์เป็น "Anyone with the link") แล้วรันโมเดล `best.pt` ตรวจจับวัตถุในวิดีโอ

In [ ]:
%pip install -q gdown

import gdown
from ultralytics import YOLO

# ดาวน์โหลดวิดีโอทดสอบจาก Google Drive
# ลิงก์ต้นฉบับ: https://drive.google.com/file/d/1jrOc0Aii6uma_UdnO1ozrBvlzkGbuOWX/view?usp=sharing
FILE_ID = "1jrOc0Aii6uma_UdnO1ozrBvlzkGbuOWX"
video_path = "/content/test_video.mp4"
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", video_path, quiet=False)

# โหลดโมเดลที่เทรนเสร็จแล้ว
model = YOLO('/content/runs/detect/train/weights/best.pt')

# รันโมเดลตรวจจับวัตถุบนวิดีโอ พร้อมบันทึกวิดีโอผลลัพธ์ (มีกรอบ + ชื่อคลาส)
model.predict(
    source=video_path,
    conf=0.5,
    imgsz=640,
    save=True,
    project="/content/runs/detect",
    name="video_predict",
    exist_ok=True
)

print("วิดีโอผลลัพธ์ถูกบันทึกไว้ที่โฟลเดอร์: /content/runs/detect/video_predict/")

In [ ]:
import glob
from IPython.display import Video

# แปลงวิดีโอผลลัพธ์เป็น MP4 (H.264) เพื่อให้แสดงผลในสมุดบันทึกนี้ได้ถูกต้อง
output_dir = "/content/runs/detect/video_predict"
input_video = (glob.glob(f"{output_dir}/*.avi") + glob.glob(f"{output_dir}/*.mp4"))[0]
output_mp4 = "/content/result_video.mp4"

!ffmpeg -y -i "{input_video}" -vcodec libx264 "{output_mp4}"

Video(output_mp4, embed=True, width=640)